# Détection automatique d’accidents anormaux à partir de données météorologiques, temporelles et spatiales

Ce notebook présente une analyse exploratoire et une détection d'anomalies sur les accidents de la route aux États-Unis, en utilisant des données météorologiques, temporelles et spatiales. Nous utilisons Spark pour le traitement des données volumineuses, puis des techniques de machine learning pour la détection d'accidents atypiques.

## 1. Import des librairies et initialisation Spark

Nous importons les librairies nécessaires pour le traitement, la visualisation et le machine learning.

In [ ]:
from pyspark.sql import SparkSession
from sklearn.preprocessing import StandardScaler
import pandas as pd
from sklearn.ensemble import IsolationForest
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
spark = SparkSession.builder\
    .master("local[*]")\
    .appName("Accident")\
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

## 2. Chargement des données

Lecture des données d'accidents depuis un fichier Parquet prétraité.

In [ ]:
data = spark.read.parquet("../../data/processed_accidents.parquet")
data.show(5, truncate=False)

## 3. Prétraitement des données

Affichage du schéma et sélection des colonnes pertinentes pour l'analyse.

In [ ]:
data.printSchema()

In [ ]:
colonnes_utiles = [
    "Severity", "Start_Time", "Distance(mi)", "Temperature(F)", "Humidity(%)",
    "Wind_Speed(mph)", "Visibility(mi)", "Weather_Condition", "Sunrise_Sunset", "Start_Lat", "Start_Lng"]
data_reduit = data.select(*colonnes_utiles)
data_sample = data_reduit.sample(fraction=0.1, seed=42)
df = data_sample.toPandas()

## 4. Analyse exploratoire et visualisations

Quelques visualisations pour mieux comprendre la distribution des accidents selon la gravité, le temps, la météo et la localisation.

In [ ]:
import plotly.express as px

sample_df = df[["Start_Lat", "Start_Lng", "Severity"]].dropna().sample(n=10000, random_state=42)

fig = px.scatter_mapbox(
    sample_df,
    lat="Start_Lat",
    lon="Start_Lng",
    color="Severity",
    color_continuous_scale="OrRd",
    zoom=3,
    height=600,
    title="Accidents de la route par gravité - USA"
)

fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(margin={"r":0,"t":50,"l":0,"b":0})
fig.show()

In [ ]:
import folium
from folium.plugins import MarkerCluster

sample_df = df[["Start_Lat", "Start_Lng", "Severity"]].dropna().sample(n=5000, random_state=42)

m = folium.Map(location=[37.0902, -95.7129], zoom_start=4, tiles="CartoDB positron")

marker_cluster = MarkerCluster().add_to(m)

for _, row in sample_df.iterrows():
    folium.Marker(
        location=[row["Start_Lat"], row["Start_Lng"]],
        popup=f"Gravité : {row['Severity']}",
        icon=folium.Icon(color="red" if row["Severity"] >= 3 else "orange")
    ).add_to(marker_cluster)

m

In [ ]:
import plotly.express as px
fig = px.histogram(df, x='Severity', nbins=4, title='Distribution de la gravité des accidents (Severity)',
                   labels={'Severity': 'Gravité (1=faible, 4=grave) '})
fig.show()

In [ ]:
# 3. Accidents par jour de la semaine
df['DayOfWeek'] = pd.to_datetime(df['Start_Time']).dt.dayofweek
jours = ['Lun', 'Mar', 'Mer', 'Jeu', 'Ven', 'Sam', 'Dim']
plt.figure(figsize=(10, 5))
sns.countplot(x=df['DayOfWeek'], palette='Set2')
plt.xticks(ticks=range(7), labels=jours)
plt.title("Répartition des accidents par jour de la semaine")
plt.xlabel("Jour")
plt.ylabel("Nombre d'accidents")
plt.tight_layout()
plt.show()

In [ ]:
# 4. Conditions météo
top_weather = df['Weather_Condition'].value_counts().nlargest(10)
plt.figure(figsize=(10, 5))
sns.barplot(x=top_weather.values, y=top_weather.index, palette="coolwarm")
plt.title("Top 10 des conditions météo lors des accidents")
plt.xlabel("Nombre d'accidents")
plt.ylabel("Condition météo")
plt.tight_layout()
plt.show()

In [ ]:
# 5. Température vs gravité
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='Severity', y='Temperature(F)', palette='pastel')
plt.title("Distribution des températures selon la gravité des accidents")
plt.xlabel("Gravité")
plt.ylabel("Température (°F)")
plt.tight_layout()
plt.show()

### Corrélations entre variables numériques

Analyse de la corrélation entre la gravité des accidents et les autres variables numériques.

In [ ]:
numerical_cols = ['Severity', 'Distance(mi)', 'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)',
                  'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)']
df_corr = data.select(numerical_cols).toPandas()
corr_matrix = df_corr[numerical_cols].corr()
plt.figure(figsize=(8, 6))
sns.barplot(x=corr_matrix['Severity'].drop('Severity').abs().sort_values(ascending=False),
            y=corr_matrix['Severity'].drop('Severity').abs().sort_values(ascending=False).index)
plt.title("Corrélation absolue avec la gravité des accidents (Severity)")
plt.xlabel("Corrélation absolue")
plt.tight_layout()
plt.show()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matrice de corrélation – Variables numériques")
plt.tight_layout()
plt.show()

## 5. Détection d'anomalies avec Isolation Forest

Nous utilisons l'algorithme Isolation Forest pour détecter les accidents atypiques en fonction de variables météorologiques, temporelles et spatiales.

In [ ]:
import os
import joblib

df_sample = data.sample(fraction=0.1, seed=42).toPandas()
features = [
    'Distance(mi)', 'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)',
    'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)',
    'Sunrise_Sunset_indexed', 'Civil_Twilight_indexed', 'Nautical_Twilight_indexed',
    'Astronomical_Twilight_indexed', 'Weather_Condition_indexed',
    'Wind_Direction_indexed', 'State_indexed', 'City_indexed'
]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_sample[features])

os.makedirs("model", exist_ok=True)
model_path = "model/isolation_forest_model.joblib"

if os.path.exists(model_path):
    model = joblib.load(model_path)
else:
    model = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
    model.fit(X_scaled)
    joblib.dump(model, model_path)

df_sample['anomaly_score'] = model.predict(X_scaled)
df_sample['is_anomaly'] = (df_sample['anomaly_score'] == -1).astype(int)
print(df_sample['is_anomaly'].value_counts())

In [ ]:
df_sample['Start_Time'] = pd.to_datetime(df_sample['Start_Time'])
df_sample['hour'] = df_sample['Start_Time'].dt.hour
plt.figure(figsize=(8, 4))
sns.countplot(data=df_sample, x='hour', hue='is_anomaly')
plt.title("Anomalies détectées par heure")
plt.xlabel("Heure")
plt.ylabel("Nombre d'accidents")
plt.legend(title="Anomalie")
plt.tight_layout()
plt.show()

### Interprétation des résultats

- La distribution des anomalies détectées par heure permet d'identifier des périodes atypiques.
- La gravité des accidents détectés comme anomalies est comparée à celle des cas normaux pour mieux comprendre leur impact.

# Prochaines étapes TODO
 - essayer plusieurs modèles de détection d'anomalies (LOF, DBSCAN, etc.)
  - deficir ce que c'est une annomalie et dire pourquoi c'est une annomalie en fonction de la gravité, du temps, de la météo, et des modeles etc

### LOF ( Detection d'anomalies locales )

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

model_path = "model/lof_model.joblib"
results_path = "model/lof_results.joblib"

if os.path.exists(model_path) and os.path.exists(results_path):
    lof_model = joblib.load(model_path)
    lof_predictions, lof_scores = joblib.load(results_path)
else:
    lof_model = LocalOutlierFactor(n_neighbors=20, contamination=0.01)
    lof_predictions = lof_model.fit_predict(X_scaled)
    lof_scores = lof_model.negative_outlier_factor_

    joblib.dump(lof_model, model_path)
    joblib.dump((lof_predictions, lof_scores), results_path)


df_sample['lof_score'] = lof_scores
df_sample['is_lof_anomaly'] = (lof_predictions == -1).astype(int)

print(df_sample['is_lof_anomaly'].value_counts())

In [ ]:
import matplotlib.patches as mpatches
plt.figure(figsize=(8,6))
plt.scatter(df_sample['Temperature(F)'], df_sample['Humidity(%)'],
            c=df_sample['is_lof_anomaly'], cmap='coolwarm', s=10)
legend_labels = [
    mpatches.Patch(color='blue', label='Normal'),
    mpatches.Patch(color='red', label='Anomalie')
]
plt.xlabel('Température (F)')
plt.ylabel('Humidité (%)')
plt.title('Anomalies LOF selon Température et Humidité')
plt.legend(handles=legend_labels)
plt.show()

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
plt.scatter(X_pca[:, 0], X_pca[:, 1],
            c=df_sample['is_lof_anomaly'], cmap='coolwarm', s=10)
legend_labels = [
    mpatches.Patch(color='blue', label='Normal'),
    mpatches.Patch(color='red', label='Anomalie')
]
plt.xlabel('Composante principale 1')
plt.ylabel('Composante principale 2')
plt.title('Anomalies LOF projetées en 2D avec PCA')
plt.legend(handles=legend_labels)
plt.show()

Evaluation avec DBSCAN

- Nous allons commencer par choisir les hypers parametres du modele DBSCAN de manière optimale avec la methode du coude (k-distance plot)

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np
import matplotlib.pyplot as plt

min_samples = 20

neigh = NearestNeighbors(n_neighbors=min_samples)
nbrs = neigh.fit(X_scaled)

distances, indices = nbrs.kneighbors(X_scaled)

k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(8, 6))
plt.plot(k_distances)
plt.ylabel(f'Distance au {min_samples}e plus proche voisin')
plt.xlabel('Points triés')
plt.title('Méthode du coude pour déterminer eps (k-distance plot)')
plt.grid()
plt.show()

L’utilisation du k-distance plot (avec min_samples = 20) a permis d’estimer une valeur optimale du paramètre eps pour l’algorithme DBSCAN. La courbe montre une progression très faible des distances jusqu’à un point de rupture net, à partir duquel les distances augmentent brutalement. Ce “coude” visuel indique la distance seuil à partir de laquelle les points deviennent de plus en plus isolés, donc moins susceptibles d’appartenir à un cluster.

`En observant cette inflexion, une valeur de eps ≈ 2.7 a été retenue comme compromis optimal entre sous-clustering (valeur trop basse) et sur-clustering (valeur trop élevée).`

In [ ]:
from sklearn.cluster import DBSCAN

model_path = "model/dbscan_model.joblib"
results_path = "model/dbscan_results.joblib"

if os.path.exists(model_path) and os.path.exists(results_path):
    dbscan_model = joblib.load(model_path)
    dbscan_predictions = joblib.load(results_path)
else:
    dbscan_model = DBSCAN(eps=3, min_samples=20)
    dbscan_predictions = dbscan_model.fit_predict(X_scaled)

    joblib.dump(dbscan_model, model_path)
    joblib.dump(dbscan_predictions, results_path)


df_sample['dbscan_labels'] = dbscan_predictions
df_sample['is_dbscan_anomaly'] = (df_sample['dbscan_labels'] == -1).astype(int)
print(df_sample['is_dbscan_anomaly'].value_counts())

In [ ]:
X_pca = PCA(n_components=2).fit_transform(X_scaled)

plt.figure(figsize=(8,6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], 
            c=df_sample['is_dbscan_anomaly'], cmap='coolwarm', s=10)

legend_labels = [
    mpatches.Patch(color='blue', label='Normal'),
    mpatches.Patch(color='red', label='Anomalie')
]
plt.legend(handles=legend_labels)
plt.xlabel('Composante principale 1')
plt.ylabel('Composante principale 2')
plt.title('Anomalies détectées par DBSCAN (projection PCA)')
plt.show()

Analyse en cluster

In [ ]:
cluster_counts = df_sample[df_sample['dbscan_labels'] != -1]['dbscan_labels'].value_counts().sort_index()
print("Nombre de points par cluster DBSCAN :\n")
print(cluster_counts)

In [ ]:
cluster_vars = [
    'Severity', 'Distance(mi)', 'Temperature(F)', 'Humidity(%)',
    'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)'
]
cluster_summary = df_sample[df_sample['dbscan_labels'] != -1].groupby('dbscan_labels')[cluster_vars].mean().round(2)

cluster_summary

In [ ]:
import matplotlib.cm as cm
import numpy as np

X_pca = PCA(n_components=2).fit_transform(X_scaled)
labels = df_sample['dbscan_labels']

plt.figure(figsize=(10,8))
unique_labels = np.unique(labels)

colors = cm.tab10(np.linspace(0, 1, len(unique_labels)))
for label, color in zip(unique_labels, colors):
    mask = (labels == label)
    label_name = f"Cluster {label}" if label != -1 else "Anomalie"
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=label_name, s=10, color=color)

plt.legend()
plt.xlabel('Composante principale 1')
plt.ylabel('Composante principale 2')
plt.title("Projection PCA des clusters DBSCAN")
plt.show()

In [ ]:
def eval_anomaly_model(y_pred, model_name="Modèle"):
    total = len(y_pred)
    anomalies = np.sum(y_pred == 1)
    normal = np.sum(y_pred == 0)
    taux_anomalie = anomalies / total * 100

    print(f"Évaluation du modèle : **{model_name}**")
    print(f"Total d'observations : {total}")
    print(f"- Anomalies détectées : {anomalies}")
    print(f"- Observations normales : {normal}")
    print(f"- Taux d’anomalies détectées : {taux_anomalie:.2f}%")
    print("—" * 40)

In [ ]:
# Isolation Forest
eval_anomaly_model(df_sample['is_anomaly'], model_name="Isolation Forest")

# LOF
eval_anomaly_model(df_sample['is_lof_anomaly'], model_name="LOF")

# DBSCAN
eval_anomaly_model(df_sample['is_dbscan_anomaly'], model_name="DBSCAN")